# Imports

In [2]:
import os
import time
import math
import json 
import random
import pickle

from datetime import timedelta
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm import tqdm

from gensim.models import FastText

from sklearn.cluster import *
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse.construct import random

from utils.model import VariationalAutoencoder
from utils.Loader import Train_Loader
from utils.tools import sanitize_string
from utils.tools import *

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/lib/python3/dist-packages/paramiko/transport.py:261: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,
/tmp/ipykernel_2997771/579917873.py:23: DeprecationWarning: Please import `random` from the `scipy.sparse` namespace; the `scipy.sparse.construct` namesp

# Loading Datasets

In [ ]:
benign_df = pd.read_pickle('dependancies/dataframe_benign.pkl')
malicious_df = pd.read_pickle('dependancies/dataframe_malicious.pkl')

In [ ]:
with open("dependancies/cadets.json", "r") as json_file:
    GT_mal = set(json.load(json_file))

malicious_processes = ['C4F4FF22-3E7B-11E8-A5CB-3FA3753A265A', '42E4A848-3E7C-11E8-A5CB-3FA3753A265A', 
                       'B6E5EA8F-3E7C-11E8-A5CB-3FA3753A265A', '17C4C0C3-3E7D-11E8-A5CB-3FA3753A265A', 
                       '7B30E76D-3E7D-11E8-A5CB-3FA3753A265A', 'BF8EF43F-3E7D-11E8-A5CB-3FA3753A265A', 
                       '219DE547-3E7E-11E8-A5CB-3FA3753A265A', '48289024-3E7E-11E8-A5CB-3FA3753A265A', 
                       'CF565BC0-3E7E-11E8-A5CB-3FA3753A265A', 'DCBB8D5C-3E7F-11E8-A5CB-3FA3753A265A', 
                       '0001A528-3E80-11E8-A5CB-3FA3753A265A', '327621AE-3E80-11E8-A5CB-3FA3753A265A', 
                       '47E61FFC-3E80-11E8-A5CB-3FA3753A265A', '47EB4608-3E80-11E8-A5CB-3FA3753A265A', 
                       'AAB27EFF-3E80-11E8-A5CB-3FA3753A265A']

# Filter DataFrame to keep rows where actorID OR objectID is in GT_mal
filtered_df = malicious_df[malicious_df['actorID'].isin(malicious_processes)]
filtered_df = filtered_df.reset_index(drop=True)

print(f"Original DataFrame: {len(malicious_df)} rows")
print(f"Filtered DataFrame: {len(filtered_df)} rows") 

# Preparing process-event-benign.txt
#### This code creates a file that has each process and the other files it interacted with in the benign dataset, so that we test an autoencoder on reconstructing their embeddings. 

In [ ]:
output_file = 'dependancies/process-event-benign.txt'

# Step 1: Build mappings
actor_info = {}  # actorID -> (exec, pid) [pid is not in your data, so we'll use actorID as a placeholder]
actor_to_files = defaultdict(set)  # actorID -> set of file paths

print("Processing DataFrame...")
for _, row in tqdm(benign_df.iterrows(), total=len(benign_df)):
    actor_id = row['actorID']
    exec_cmd = row['exec'].lower()  # Using 'exec' as the command line
    filepath = row['path'].lower()  # Using 'path' as the file path

    # Store actor info (exec and a placeholder PID)
    if actor_id not in actor_info:
        actor_info[actor_id] = (exec_cmd, str(actor_id))  # Using actorID as a fake PID

    # Track files accessed by this actor (exec)
    actor_to_files[actor_id].add(filepath)

# Step 2: Write to output file
print(f"Writing output to {output_file}...")
with open(output_file, 'w') as out:
    for actor_id, (exec_cmd, pid) in actor_info.items():
        if actor_id not in actor_to_files:
            continue  # Skip if no paths interacted with

        # Write process header line: exec$$$pid$$$false
        out.write(f"{exec_cmd}$$${pid}$$$false")

        # Write up to 6 associated paths
        file_list = list(actor_to_files[actor_id])
        for fpath in file_list:
            out.write(f"{fpath}\n")

        # Blank line separator
        out.write("\n")

print("Done.") 

# Preparing process-event-anomaly.txt
#### This code creates a file that has each process and the other files it interacted with in the malicious dataset, so that we test an autoencoder on reconstructing their embeddings. 

In [ ]:
output_file = 'dependancies/process-event-anomaly.txt'

# Step 1: Build mappings
actor_info = {}  # actorID -> (exec, pid) [pid is not in your data, so we'll use actorID as a placeholder]
actor_to_files = defaultdict(set)  # actorID -> set of file paths

print("Processing DataFrame...")
for _, row in tqdm(filtered_df.iterrows(), total=len(filtered_df)):
    actor_id = row['actorID']
    exec_cmd = row['exec'].lower()  # Using 'exec' as the command line
    filepath = row['path'].lower()  # Using 'path' as the file path

    # Store actor info (exec and a placeholder PID)
    if actor_id not in actor_info:
        actor_info[actor_id] = (exec_cmd, str(actor_id))  # Using actorID as a fake PID

    # Track files accessed by this actor (exec)
    actor_to_files[actor_id].add(filepath)

# Step 2: Write to output file
print(f"Writing output to {output_file}...")
with open(output_file, 'w') as out:
    for actor_id, (exec_cmd, pid) in actor_info.items():
        if actor_id not in actor_to_files:
            continue  # Skip if no paths interacted with

        # Write process header line: exec$$$pid$$$false
        out.write(f"{exec_cmd}$$${pid}$$$true")

        # Write up associated paths
        file_list = list(actor_to_files[actor_id])
        for fpath in file_list:
            out.write(f"{fpath}\n")

        # Blank line separator
        out.write("\n")

print("Done.") 

# Embedding functions
#### Used in computing the embedding of each process using FastText.

In [ ]:
def extract_process_feature_benign(file_path, w2v):

    f = open(file_path, 'r')
    id = 0  # This is used as a unique process ID, incremented every time we hit a blank line (i.e., end of one process block)

    file_freq = defaultdict(set)  # Keeps track of which process IDs each file appears in
    file_vec = []  # List to store the embedding vector for each unique file path
    fileid2name = []  # List to store the canonical name (string) for each unique file path

    while True:
        line = f.readline()

        # If it's a blank line, increment the process ID
        if line == '\n':
            id += 1

        # End of file
        if not line:
            break

        # Clean the line (remove trailing newline, lowercase)
        filepath = line.strip().lower()

        # Skip lines that end with $$$true or $$$false (used for ground truth labeling elsewhere)
        if filepath.endswith('$$$true') or filepath.endswith('$$$false'):
            continue

        # Tokenize the file path into components using sanitize_string (e.g., split by slashes, remove punctuation)
        split_path = sanitize_string(filepath)

        # Create a normalized string version of the path for indexing
        newname = '/'.join(split_path)

        # Skip empty results (e.g., if path couldn't be tokenized)
        if len(split_path) == 0:
            continue

        # If this file has not been seen before, compute its embedding
        if newname not in fileid2name:
            tmp = []
            for token in split_path:
                tmp += [w2v.wv[token]]  # Get the FastText embedding for each token
            r = np.mean(tmp, axis=0).tolist()  # Take the mean of all token vectors to represent the full path
            file_vec.append(r)  # Save the embedding
            fileid2name.append(newname)  # Save the canonical name

        # Record that this file appears in this process ID
        file_freq[newname].add(id)

    return file_vec, fileid2name, file_freq, id


In [18]:
def extract_process_feature_anomaly(file_path,tfidf, stability, w2v, c2v):
    process_map = {}
    f = open(file_path,'r')
    process_vec = defaultdict(list)
    id = 0
    mean_s = np.mean(list(tfidf.values()))
    max_s = np.max(list(tfidf.values()))
    isprocess_file = True
    tmp_process_vec = []
    cmdline_vec = []
    ground_truth = {}
    while True:
        line = f.readline()
        if line == '\n':
            process_vec[id] = np.mean(tmp_process_vec,axis=0).tolist()

            id += 1
            tmp_process_vec = []
            cmdline_vec = []
            isprocess_file = True
            continue
        if not line:
            break

        filepath = line.strip().lower()
        if filepath.endswith('$$$true'):
            filepath = filepath.replace('$$$true','')
            ground_truth[id] = filepath
        else:
            filepath = filepath.replace('$$$false','')

        if '$$$' in filepath:
            filepath, pname = filepath.split('$$$')[0], filepath.split('$$$')[1]


        split_path = sanitize_string(filepath)
        if len(split_path) == 0:
            continue
        if isprocess_file:
            process_map[id] = pname
            isprocess_file = False
            tmp = []
            for l,i in enumerate(split_path):
                tmp += [c2v.wv[i]]
            r = np.mean(tmp,axis=0)
            r = r * mean_s
            
        else:
            tmp = []
            for l,i in enumerate(split_path):
                tmp += [w2v.wv[i]]
            r = np.mean(tmp,axis=0)
            newname = '/'.join(split_path)
            if newname in tfidf:
                s = tfidf[newname]
            else:
                s = mean_s
            r = r * s

        tmp_process_vec.append(r.tolist())
    return process_vec, process_map, ground_truth


# Loading detection model (VAE)

In [ ]:
w2v_dic = 'dependancies/filepath-embedding_training.model'
w2v = FastText.load(w2v_dic)
c2v_dic = 'dependancies/cmdline-embedding_training.model'
c2v = FastText.load(c2v_dic)
benign_data_file = 'dependancies/process-event-benign-training.txt'
tfidf_file = 'dependancies/tfidf.json'
tfidf_dic = json.load(open(tfidf_file))
stability_file = 'dependancies/stability-embedding.json'
stability = json.load(open(stability_file))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VariationalAutoencoder(32)
model.load_state_dict(torch.load('dependancies/AE.model'))
model.to(device)
model.eval()

criterion = nn.MSELoss(reduction='sum')

valid_data = json.load(open('dependancies/process_embedding_train.json'))

# We had a problem with the way threshold was computed (code in train notebook), so we set it manually to 130 based on the distribution of losses we observed.
# We kept the code for threshold computation in train notebook for reference
anomaly_cutoff = 130  

# Testing on malicious dataset

In [ ]:
# ---------------------------
# CONFIG
# ---------------------------
VAE_RETURNS_TUPLE = False          

ANOMALY_FILE = 'dependancies/process-event-anomaly.txt'
BENIGN_FILE  = 'dependancies/process-event-benign.txt'

# ---------------------------
# HELPERS
# ---------------------------

def forward_reconstruct(x_tensor):
    if VAE_RETURNS_TUPLE:
        recon, mu, logvar = model(x_tensor)
        return recon
    else:
        return model(x_tensor)

def safe_stability_norm(loss_value, proc_name, stability_dict):
    if proc_name in stability_dict:
        s = stability_dict[proc_name]
        if s is not None and s > 0:
            return loss_value / (math.log(s) + 1.0)
    return loss_value

def compute_metrics(TP, FP, FN, TN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # TPR
    TNR       = TN / (TN + FP) if (TN + FP) > 0 else 0.0  # specificity
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / max(TP + FP + FN + TN, 1)
    return precision, recall, TNR, f1, accuracy

def build_benign_process_vectors(file_vec, fileid2name, file_freq, n_proc):
    """
    Aggregate file embeddings into process-level vectors.
    """
    proc_vecs = {}
    for pid in range(n_proc):
        this_proc_vecs = []
        for file, procs in file_freq.items():
            if pid in procs:
                idx = fileid2name.index(file)
                this_proc_vecs.append(file_vec[idx])
        if this_proc_vecs:
            proc_vecs[pid] = np.mean(this_proc_vecs, axis=0).tolist()
    return proc_vecs

# ---------------------------
# DATA: extract features
# ---------------------------
anom_vec, anom_map, attack_process = extract_process_feature_anomaly(
    ANOMALY_FILE, tfidf_dic, stability, w2v, c2v
) 

file_vec, fileid2name, file_freq, n_proc = extract_process_feature_benign( 
    BENIGN_FILE, w2v
)

# build process-level benign vectors
benign_vec = build_benign_process_vectors(file_vec, fileid2name, file_freq, n_proc)

# unify to one evaluation set
all_vec         = {**benign_vec, **anom_vec}
all_process_map = {**anom_map}  # benign_map not available (just use PID index as name)
mal_set         = set(attack_process.keys()) if not isinstance(attack_process, set) else attack_process
ben_set         = set(benign_vec.keys())

# ---------------------------
# INFERENCE: compute loss per PID
# ---------------------------
model.eval()
pid_list   = []
loss_list  = []

with torch.no_grad():
    for pid, feat in all_vec.items():
        try:
            x = torch.as_tensor(feat, dtype=torch.float32, device=device)
            if x.ndim == 1:
                x_in = x.view(1, -1)
            else:
                x_in = x.view(1, -1)

            recon = forward_reconstruct(x_in)
            loss  = criterion(x_in, recon).item()

            name = all_process_map.get(pid, 'None')
            norm_loss = safe_stability_norm(loss, name, stability)

            pid_list.append(int(pid))
            loss_list.append(norm_loss)
        except Exception:
            continue

print(f"Total processes evaluated: {len(pid_list)}")

# ---------------------------
# CLASSIFY via cutoff
# ---------------------------
TP = FP = FN = TN = 0

for pid, loss in zip(pid_list, loss_list):
    is_mal = pid in mal_set
    is_ben = pid in ben_set

    pred_mal = (loss > anomaly_cutoff)

    if is_mal:
        if pred_mal: TP += 1
        else: FN += 1
    elif is_ben:
        if pred_mal: FP += 1
        else: TN += 1
    else:
        # default: treat unknowns as benign
        if pred_mal: FP += 1
        else: TN += 1

# ---------------------------
# METRICS
# ---------------------------
precision, TPR, TNR, F1, ACC = compute_metrics(TP, FP, FN, TN)

print("-------- Results @ cutoff =", anomaly_cutoff, "--------")
print(f"TP: {TP} | FP: {FP} | FN: {FN} | TN: {TN}")
print(f"Precision:       {precision:.4f}")
print(f"Recall (TPR):    {TPR:.4f}")
print(f"TNR (Specific.): {TNR:.4f}")
print(f"F1 Score:        {F1:.4f}")
print(f"FPR:             {FP / (FP + TN):.4f}")


Total processes evaluated: 28622
-------- Results @ cutoff = 130 --------
TP: 15 | FP: 17 | FN: 0 | TN: 28590
Precision:       0.4688
Recall (TPR):    1.0000
TNR (Specific.): 0.9994
F1 Score:        0.6383
FPR:             0.0006


# Contorter
#### Here we start applying steps of Contorter to curate the malicious dataset

# 1- Group Benign Candidates By labels

In [ ]:
# This step is skipped as we are hiding processes only because NodLink detects malicious processes only. 

# 2- Filter candidate nodes by number of interactions

In [14]:
# File paths
input_file = "dependancies/process-event-benign.txt"
output_file = "dependancies/candidates_filtered_by_interactions.txt"

F_min = 50
F_max = 100 

with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
    current_command = None
    interaction_lines = []

    for line in fin:
        line = line.strip()

        if not line:
            continue  # skip blank lines

        if '$$$' in line:
            # New command detected: handle previous block
            if current_command is not None:
                if F_min <= len(interaction_lines) <= F_max:
                    fout.write(current_command + '\n')
                    # for i in range(5):
                    for interaction in interaction_lines:
                            fout.write(interaction + '\n')
                    fout.write('\n')  # separate each process block with a newline

            current_command = line
            interaction_lines = []
        else:
            interaction_lines.append(line)

    # Handle last block at EOF
    if current_command and F_min <= len(interaction_lines) <= F_max:
        fout.write(current_command + '\n')
        for interaction in interaction_lines:
            fout.write(interaction + '\n')
        fout.write('\n')


# 3- Filter based on similarity

In [ ]:
def parse_process_event_file(path):
    """Parses a process-event file into a list of (cmdline, [files...]) pairs."""
    processes = []
    with open(path, 'r') as f:
        current_cmd = None
        current_files = []
        for line in f:
            line = line.strip()
            if '$$$' in line:
                if current_cmd is not None:
                    processes.append((current_cmd, current_files))
                current_cmd = line
                current_files = []
            elif line:
                current_files.append(line)
        if current_cmd:
            processes.append((current_cmd, current_files))
    return processes

def vectorize_command(cmdline, model):
    """Generates an embedding vector for the command line using the provided model."""
    tokens = cmdline.lower().split()
    vectors = [model.wv[tok] for tok in tokens if tok in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

# Parse input files
malicious_procs = parse_process_event_file('dependancies/process-event-anomaly.txt')
benign_candidates = parse_process_event_file('dependancies/candidates_filtered_by_interactions.txt')

# Vectorize commands
mal_vecs = [vectorize_command(cmd, c2v) for cmd, _ in malicious_procs]
benign_vecs = [vectorize_command(cmd, c2v) for cmd, _ in benign_candidates]

# Compute cosine similarity
similarities = cosine_similarity(mal_vecs, benign_vecs)

# Create output directory
os.makedirs('dependancies/candidates', exist_ok=True)

# Select top 10% most similar benign candidates for each malicious process
num_candidates = len(benign_candidates)
top_k = max(1, int(num_candidates * 0.10))  # At least one

for i, sim_row in enumerate(similarities):
    # Get indices of top 10% similar benign candidates
    top_indices = np.argsort(sim_row)[-top_k:][::-1]  # Descending order

    # Write to a file named "1.txt", "2.txt", etc.
    with open(f'dependancies/candidates/{i + 1}.txt', 'w') as fout:
        for idx in top_indices:
            cmdline, files = benign_candidates[idx]
            fout.write(cmdline + '\n')
            for f in files:
                fout.write(f + '\n')
            fout.write('\n')


# 4- Filter based on confidence 

In [ ]:
def parse_single_candidate_file(path):
    """Parses one candidate file into a list of (cmdline, [files...]) pairs."""
    return parse_process_event_file(path)

def vectorize_command(cmdline, model):
    """Generates an embedding vector for the command line using the provided model."""
    tokens = cmdline.lower().split()
    vectors = [model.wv[tok] for tok in tokens if tok in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

# Load already-augmented malicious processes
malicious_procs = parse_process_event_file('dependancies/process-event-anomaly.txt')

# Directory where filtered candidate files are stored
candidate_dir = 'dependancies/candidates'

# For each malicious process, find the lowest-loss candidate from its filtered list
final_augmented = []
for i, (mal_cmd, mal_files) in enumerate(tqdm(malicious_procs, desc="Refining with lowest-loss per-malicious candidate")):
    candidate_path = os.path.join(candidate_dir, f"{i + 1}.txt")
    if not os.path.exists(candidate_path):
        print(f"Warning: {candidate_path} not found. Skipping.")
        final_augmented.append((mal_cmd, mal_files))
        continue

    candidates = parse_single_candidate_file(candidate_path)
    
    best_loss = float('inf')
    best_files = []

    for cmd, files in candidates:
        vec = vectorize_command(cmd, c2v)
        input_tensor = torch.tensor(vec, dtype=torch.float32).to(device)
        output = model(input_tensor)
        loss = criterion(output, input_tensor).item()

        if loss < best_loss:
            best_loss = loss
            best_files = files

    # Append best candidate interactions 
    augmented_files = mal_files + best_files 
    final_augmented.append((mal_cmd, augmented_files))

# Save the final augmented malicious processes
with open('dependancies/final-augmented-malicious-processes.txt', 'w') as fout:
    for cmd, files in final_augmented:
        fout.write(cmd + '\n')
        for f in files:
            fout.write(f + '\n')
        fout.write('\n')


Refining with lowest-loss per-malicious candidate: 100%|██████████| 15/15 [00:00<00:00, 98.26it/s] 


# Test Evasion
### Using most confident node among top 10% similar nodes

In [1]:
# ---------------------------
# CONFIG
# ---------------------------
VAE_RETURNS_TUPLE = False          # set True if model(x) -> (recon, mu, logvar)

ANOMALY_FILE = 'dependancies/final-augmented-malicious-processes.txt'
BENIGN_FILE  = 'dependancies/process-event-benign.txt'

# ---------------------------
# HELPERS
# ---------------------------

def forward_reconstruct(x_tensor):
    if VAE_RETURNS_TUPLE:
        recon, mu, logvar = model(x_tensor)
        return recon
    else:
        return model(x_tensor)

def safe_stability_norm(loss_value, proc_name, stability_dict):
    if proc_name in stability_dict:
        s = stability_dict[proc_name]
        if s is not None and s > 0:
            return loss_value / (math.log(s) + 1.0)
    return loss_value

def compute_metrics(TP, FP, FN, TN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # TPR
    TNR       = TN / (TN + FP) if (TN + FP) > 0 else 0.0  # specificity
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / max(TP + FP + FN + TN, 1)
    return precision, recall, TNR, f1, accuracy

def build_benign_process_vectors(file_vec, fileid2name, file_freq, n_proc):
    """
    Aggregate file embeddings into process-level vectors.
    """
    proc_vecs = {}
    for pid in range(n_proc):
        this_proc_vecs = []
        for file, procs in file_freq.items():
            if pid in procs:
                idx = fileid2name.index(file)
                this_proc_vecs.append(file_vec[idx])
        if this_proc_vecs:
            proc_vecs[pid] = np.mean(this_proc_vecs, axis=0).tolist()
    return proc_vecs

# ---------------------------
# DATA: extract features
# ---------------------------
anom_vec, anom_map, attack_process = extract_process_feature_anomaly(
    ANOMALY_FILE, tfidf_dic, stability, w2v, c2v
) 

file_vec, fileid2name, file_freq, n_proc = extract_process_feature_benign( 
    BENIGN_FILE, w2v
)

# build process-level benign vectors
benign_vec = build_benign_process_vectors(file_vec, fileid2name, file_freq, n_proc)

# unify to one evaluation set
all_vec         = {**benign_vec, **anom_vec}
all_process_map = {**anom_map}  # benign_map not available (just use PID index as name)
mal_set         = set(attack_process.keys()) if not isinstance(attack_process, set) else attack_process
ben_set         = set(benign_vec.keys())

# ---------------------------
# INFERENCE: compute loss per PID
# ---------------------------
model.eval()
pid_list   = []
loss_list  = []

with torch.no_grad():
    for pid, feat in all_vec.items():
        try:
            x = torch.as_tensor(feat, dtype=torch.float32, device=device)
            if x.ndim == 1:
                x_in = x.view(1, -1)
            else:
                x_in = x.view(1, -1)

            recon = forward_reconstruct(x_in)
            loss  = criterion(x_in, recon).item()

            name = all_process_map.get(pid, 'None')
            norm_loss = safe_stability_norm(loss, name, stability)

            pid_list.append(int(pid))
            loss_list.append(norm_loss)
        except Exception:
            continue

print(f"Total processes evaluated: {len(pid_list)}")

# ---------------------------
# CLASSIFY via cutoff
# ---------------------------
TP = FP = FN = TN = 0

for pid, loss in zip(pid_list, loss_list):
    is_mal = pid in mal_set
    is_ben = pid in ben_set

    pred_mal = (loss > anomaly_cutoff)

    if is_mal:
        if pred_mal: TP += 1
        else: FN += 1
    elif is_ben:
        if pred_mal: FP += 1
        else: TN += 1
    else:
        # default: treat unknowns as benign
        if pred_mal: FP += 1
        else: TN += 1

# ---------------------------
# METRICS
# ---------------------------
precision, TPR, TNR, F1, ACC = compute_metrics(TP, FP, FN, TN)

print("-------- Results @ cutoff =", anomaly_cutoff, "--------")
print(f"TP: {TP} | FP: {FP} | FN: {FN} | TN: {TN}")
print(f"Precision:       {precision:.4f}")
print(f"Recall (TPR):    {TPR:.4f}")
print(f"TNR  {TNR:.4f}")
print(f"F1 Score:        {F1:.4f}")
print(f"FPR:             {FP / (FP + TN):.4f}")


NameError: name 'extract_process_feature_anomaly' is not defined